# 00 | Systemtest: Markov-Ketten in JupyterLite

**Kieler Woche der Mathematik · Einführung**

Dieses Notebook ist gleichzeitig ein technischer Funktionstest und ein erstes Beispiel. Arbeite die Zellen von oben nach unten durch. Klicke eine Codezelle an und führe sie mit **Shift + Enter** oder ▶ aus. Beim ersten Start muss der Python-Kernel möglicherweise noch aus dem Internet geladen werden.

**Wichtig:** Deine Änderungen werden im Browser gespeichert, nicht automatisch auf einen zentralen Server. Lade deine Arbeit zum Sichern als `.ipynb` herunter (z. B. über **File → Download** oder Rechtsklick auf die Datei → **Download**).

## 1. Funktioniert Python?
Führe folgende Zelle aus. Sie sollte `Python funktioniert: 4` ausgeben.

In [ ]:
print("Python funktioniert:", 2 + 2)

## 2. Übergangsmatrizen und Zwei-Schritt-Wahrscheinlichkeiten

Wir betrachten eine Markov-Kette auf $\{0,1\}$ mit Übergangsmatrix

$$P=\begin{pmatrix}0.8&0.2\\0.3&0.7\end{pmatrix}.$$

Dabei ist $P_{ij}=\mathbb P(X_{t+1}=j\mid X_t=i)$. Der Eintrag $(P^2)_{01}$ ist die Wahrscheinlichkeit, aus Zustand 0 nach zwei Schritten in Zustand 1 zu sein.

In [ ]:
import numpy as np

P = np.array([[0.8, 0.2],
              [0.3, 0.7]])

P2 = np.linalg.matrix_power(P, 2)
print("P² =\n", P2)
print("P(X₂ = 1 | X₀ = 0) =", P2[0, 1])
assert np.allclose(P2, [[0.70, 0.30], [0.45, 0.55]])
print("Matrixrechnung funktioniert.")

**Zum Ausprobieren:** Ändere in der vorigen Zelle die Potenz `2` zu `5`. Was bedeutet der Eintrag `[0, 1]` dann? Welche Eigenschaft muss jede Zeile der neuen Matrix haben?

## 3. Monte-Carlo-Simulation

Wir beginnen in Zustand 0, simulieren $N$ Zeitpunkte $X_0,\dots,X_{N-1}$ und zählen die Aufenthalte in den beiden Zuständen. Die berechneten relativen Häufigkeiten sind Zufallsgrößen.

In [ ]:
rng = np.random.default_rng(2026)
N = 10_000
x = 0
haeufigkeiten = np.zeros(2, dtype=int)

for _ in range(N):
    haeufigkeiten[x] += 1
    x = rng.choice(2, p=P[x])

zeitanteile = haeufigkeiten / N
print("Besuche:", haeufigkeiten)
print("Zeitanteile:", zeitanteile)
print("Summe der Zeitanteile:", zeitanteile.sum())

**Zum Ausprobieren:** Wiederhole die Simulation für `N = 100`, `N = 1000` und `N = 100000`. Bleiben die Ergebnisse bei jedem Durchlauf gleich? Was passiert, wenn du in `np.random.default_rng(2026)` die Zahl `2026` veränderst?

## 4. Vergleich mit der stationären Verteilung

Für $\pi=(\pi_0,\pi_1)$ gelten $\pi P=\pi$ und $\pi_0+\pi_1=1$. Hier folgt aus $0.2\pi_0=0.3\pi_1$ die stationäre Verteilung $\pi=(0.6,0.4)$.

In [ ]:
pi = np.array([0.6, 0.4])
assert np.allclose(pi @ P, pi)
print("Theoretische Verteilung:", pi)
print("Simulierte Zeitanteile:", zeitanteile)
print("Absolute Abweichungen:", np.abs(zeitanteile - pi))

## 5. Grafische Darstellung (optional)

Die folgende Zelle prüft auch, ob `matplotlib` funktioniert. Der erste Import kann etwas länger dauern.

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)
M = 3000
x = 0
besuche_null = 0
verlauf = []
for t in range(1, M + 1):
    besuche_null += (x == 0)
    verlauf.append(besuche_null / t)
    x = rng.choice(2, p=P[x])

plt.figure(figsize=(8, 3))
plt.plot(range(1, M + 1), verlauf, label="Empirischer Zeitanteil von Zustand 0")
plt.axhline(0.6, linestyle="--", label="Stationärer Anteil 0,6")
plt.xlabel("Anzahl beobachteter Zeitpunkte")
plt.ylabel("Zeitanteil")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Zusatzaufgabe für Fortgeschrittene

Ersetze die Matrix $P$ durch

$$Q=\begin{pmatrix}0&1\\1&0\end{pmatrix}.$$

Die Kette springt deterministisch zwischen 0 und 1 hin und her. Untersuche zwei **verschiedene** Aussagen: Konvergiert die Verteilung von $X_n$ bei Start $X_0=0$? Konvergieren die Zeitanteile? Begründe beide Antworten ohne Simulation.